# Getting Started with Spicy Regs

Three questions come up often:

1. **How do I access the data directly?**
2. **How do I segment, search, and filter it?**
3. **How do I build and save a dataset for research or a product?**

This notebook answers all three end to end, using only `duckdb` + `pandas` — no server, no credentials, no downloads of multi-GB files.

For depth on any one piece, jump to a dedicated notebook instead of duplicating it here:

- [`search_capabilities.ipynb`](search_capabilities.ipynb) — the three search layers, including the canonical-vs-legacy comment partition trap in detail.
- [`data_explorer.ipynb`](data_explorer.ipynb) — full schema reference, agency explorer, export helpers.
- [`iceberg_explorer.ipynb`](iceberg_explorer.ipynb) — the R2 Data Catalog directly (maintainer credentials required).


## Part 1 — Access: four doors into the data

All four read the *same* public corpus. Pick the one that fits the tool you already have open.

| Door | What it is | Needs credentials? |
|---|---|---|
| **DuckDB over HTTP** | `read_parquet('https://data.spicy-regs.dev/...')` from any Python/SQL shell | No |
| **MCP server** | `list_sources` / `describe_table` / `query_sql` tools at `https://mcp.spicy-regs.dev/mcp`, for Claude and other MCP clients | No |
| **CLI** | `spicy-regs download\|stats\|sample\|search\|agencies` (installed with the `spicy-regs` package) | No |
| **Docs site** | The generated data dictionary — schema + row counts + coverage notes per table | No (just a browser) |

This notebook uses the first door throughout, since it's the one a Jupyter session can exercise directly.

### Door 1 — DuckDB over HTTP

The pattern every cell below builds on.

In [1]:
import duckdb

R2_BASE_URL = "https://data.spicy-regs.dev"

conn = duckdb.connect()
conn.execute("INSTALL httpfs; LOAD httpfs;")
conn.execute("PRAGMA disable_progress_bar")

# Proof of life: row counts straight off the public corpus, no local files.
conn.execute(f"""
    SELECT COUNT(*) AS dockets FROM read_parquet('{R2_BASE_URL}/dockets.parquet')
""").fetchdf()

,dockets
0,276326


**One hard constraint worth internalizing now:** R2's public HTTPS endpoint has no object listing, so DuckDB can't expand a glob like `comments/**/*.parquet` here — that 404s. Every path below is either a single named file, or a file discovered through a small index/rollup queried first. Wildcard globs only work over `s3://` with R2 credentials (maintainer-only, see `iceberg_explorer.ipynb`).

### Door 2 — the MCP server

If you're working from Claude (or another MCP client) rather than a notebook, the hosted server at `https://mcp.spicy-regs.dev/mcp` exposes three tools backed by the exact same DuckDB-over-Parquet engine used below:

- `list_sources()` — the published tables
- `describe_table(table)` — schema + row count for one table
- `query_sql(sql, max_rows=25)` — arbitrary read-only SQL, capped rows, ~13-minute statement timeout

Locally it also runs over stdio: `uv run spicy-regs-mcp` (from a `spicy-regs/` checkout). Nothing to run here — it's a protocol for an AI client, not a Python call — but it's the door to reach for when you want an agent to query this data on your behalf instead of writing the SQL yourself.

### Door 3 — the CLI

Installed alongside the `spicy-regs` package. `--help` costs nothing over the network, so it's safe to run here:

In [2]:
# Run from a spicy-regs/ checkout (this notebook lives in notebooks/, one level down).
!uv run --project .. spicy-regs --help

usage: spicy-regs [-h] [--output-dir OUTPUT_DIR]
                  {download,stats,sample,search,agencies} ...

Download and explore federal regulations data from Spicy Regs

positional arguments:
  {download,stats,sample,search,agencies}
                        Available commands
    download            Download parquet files
    stats               Show dataset statistics
    sample              Show sample rows
    search              Search across datasets
    agencies            List all agencies

options:
  -h, --help            show this help message and exit
  --output-dir OUTPUT_DIR, -o OUTPUT_DIR
                        Output directory for data files (default: spicy-regs-
                        data)


`spicy-regs sample <table>`, `search "<query>"`, and `agencies` all read from a *local* copy that `spicy-regs download` fetches first — that's a heavier, download-everything-then-query workflow, the opposite of the stream-only-what-you-need approach this notebook uses. Reach for the CLI when you want a quick terminal answer without opening Python at all; reach for DuckDB-over-HTTP (or the MCP server) when you're building something.

### Door 4 — the docs site

The generated data dictionary (schema, row counts, per-table coverage caveats, cross-source join keys) is published from `spicy_regs.data_dictionary` — see the project docs. Best door when you just need "what columns does `X` have" without spinning up DuckDB at all.

## Part 2 — Segment & filter: rollups first, then partitions

The corpus is **20 published tables**, not just `dockets` / `documents` / `comments`. Most questions about *aggregates* — top agencies, busiest dockets, monthly volume — are already answered by a small, denormalized rollup that's meant to be read whole. Reach for the base tables only when you need row-level detail the rollup doesn't carry.

In [3]:
# agency_stats: one row per agency, meant to be read whole. Compare this to
# aggregating comments.parquet yourself (also fine here, since it's a single
# GROUP BY DuckDB can push down — but for a 20-table corpus, checking for a
# rollup first is the habit that scales).
conn.execute(f"""
    SELECT agency_code, docket_count, document_count, comment_count
    FROM read_parquet('{R2_BASE_URL}/agency_stats.parquet')
    ORDER BY comment_count DESC
    LIMIT 10
""").fetchdf()

,agency_code,docket_count,document_count,comment_count
0,FWS,2294,15995,2629148
1,FDA,64140,238651,1801740
2,CMS,4086,7949,1420828
3,EPA,21443,525834,1133975
4,HHS,807,3188,1105806
5,CFPB,752,2220,1051826
6,CEQ,73,186,1025767
7,ED,2795,9869,1013324
8,ATF,30,111,1005860
9,BLM,82,455,862270


In [4]:
# feed_summary: one row per docket, comment counts pre-joined in. This is
# what the site's /feed page reads — no join against comments.parquet needed
# just to rank dockets by activity.
conn.execute(f"""
    SELECT docket_id, title, comment_count, comment_end_date
    FROM read_parquet('{R2_BASE_URL}/feed_summary.parquet')
    WHERE agency_code = 'EPA'
    ORDER BY comment_count DESC
    LIMIT 5
""").fetchdf()

,docket_id,title,comment_count,comment_end_date
0,EPA-HQ-OA-2017-0190,Evaluation of Existing Regulations,63411,2017-05-16T03:59:59Z
1,EPA-HQ-OAR-2025-0194,Rescission of the Greenhouse Gas Endangerment ...,30933,2025-09-23T03:59:59Z
2,EPA-HQ-OAR-2017-0355,Repeal of Carbon Dioxide Emission Guidelines f...,26206,2018-11-01T03:59:59Z
3,EPA-HQ-OA-2018-0259,Strengthening Transparency in Regulatory Science,22390,2020-05-19T03:59:59Z
4,EPA-HQ-OAR-2006-0173,California State Motor Vehicle Pollution Contr...,19452,2009-04-07T03:59:59Z


### Reading comments at scale: partition-aware, not the flat file

`comments.parquet` (~2.5 GB, 25M+ rows) is a real file and a valid target for one-off full-corpus analytics, but it's the wrong tool for "give me this docket's comments" — you'd be streaming past everything to find a few hundred rows.

Comments are also mirrored **partitioned by agency**: one Parquet file per agency at `comments/agency/agency_code={X}/part-0.parquet`, rebuilt daily, sorted by `(docket_id, posted_date)`. Scope to one agency (and ideally a `docket_id`) and DuckDB prunes the rest via Parquet row-group stats.

*(There's also an older `comments/agency_code={A}/docket_id={D}/year={Y}/month={M}/...` tree from before the ETL moved comments onto an Iceberg catalog — it stopped being written to in mid-April 2026, so building URLs from it silently 404s on anything created since. See `search_capabilities.ipynb` for the full story; use the per-agency tree below instead.)*

In [5]:
# Docket-scoped read against the canonical per-agency tree — fast even
# though the EPA file alone is ~300 MB, because the docket_id filter prunes
# row groups instead of scanning the whole file.
conn.execute(f"""
    SELECT comment_id, posted_date, organization, LEFT(comment, 120) AS preview
    FROM read_parquet('{R2_BASE_URL}/comments/agency/agency_code=EPA/part-0.parquet')
    WHERE docket_id = 'EPA-HQ-OAR-2021-0317'
    ORDER BY posted_date DESC
    LIMIT 5
""").fetchdf()

,comment_id,posted_date,organization,preview
0,EPA-HQ-OAR-2021-0317-4068,2025-07-23T04:00:00Z,None,See Attached
1,EPA-HQ-OAR-2021-0317-4067,2024-10-23T04:00:00Z,None,See Attached
2,EPA-HQ-OAR-2021-0317-4066,2024-09-09T04:00:00Z,None,The Wyoming Department of Environmental Qualit...
3,EPA-HQ-OAR-2021-0317-4065,2024-09-04T04:00:00Z,None,The American Petroleum Institute (API) respect...
4,EPA-HQ-OAR-2021-0317-4064,2024-08-28T04:00:00Z,None,"Public comment submission from Panna Chibber, ..."


*Caveat while we're in the comments schema: `attachments_json` is populated on only ~0.2% of comments corpus-wide, and `text_extraction_status` is NULL on ~99.8% of comments (100% of documents). Attachment text is not yet a reliable field to filter or search on — `comment`/`title` are still the primary searchable text.*

### Cross-source join: docket → RIN → Unified Agenda

Beyond the regulations.gov mirror, the corpus includes federal sources joined on shared keys — `rin` (Regulation Identifier Number) links `unified_agenda` and `federal_register`; `uei` links `sam_entities` and `usaspending_recipients`; CFR citations and `agency_code` cut across most of the rest.

`fr_docket_links` is the bridge from a `docket_id` to its Federal Register document(s), which carry the RIN(s). From there, `unified_agenda` gives you the rulemaking's stage and next-action date — context regulations.gov itself doesn't surface.

In [6]:
# Which EPA dockets have an open Unified Agenda entry, and what stage is it at?
conn.execute(f"""
    WITH docket_rins AS (
        SELECT DISTINCT docket_id,
               TRIM(UNNEST(regulation_id_numbers_json::JSON[])::VARCHAR, '"') AS rin
        FROM read_parquet('{R2_BASE_URL}/fr_docket_links.parquet')
        WHERE docket_id LIKE 'EPA-%'
    )
    SELECT dr.docket_id, dr.rin, ua.rule_stage, ua.next_action_date, ua.title
    FROM docket_rins dr
    JOIN read_parquet('{R2_BASE_URL}/unified_agenda.parquet') ua USING (rin)
    ORDER BY ua.next_action_date DESC NULLS LAST
    LIMIT 5
""").fetchdf()

,docket_id,rin,rule_stage,next_action_date,title
0,EPA-HQ-OAR-2002-0037,2060-AR73,Long-Term Actions,2030-09-01,National Emission Standards for Hazardous Air ...
1,EPA-HQ-OAR-2023-0509,2060-AW16,Long-Term Actions,2028-06-01,Removal of Affirmative Defense Provisions from...
2,EPA-HQ-OPPT-2023-0223,2070-AL40,Proposed Rule Stage,2027-08-01,Laboratory Requirements under the Toxic Substa...
3,EPA-HQ-OAR-2025-3028,2060-AW94,Proposed Rule Stage,2027-07-01,Federal Plan Requirements for Other Solid Wast...
4,EPA-HQ-OW-2024-0592,2040-AG36,Long-Term Actions,2027-07-01,National Primary Drinking Water Regulation for...


## Part 3 — Build & save: turn a query into a dataset

Two ways to materialize a result, both shown below:

- **`.df()` in pandas** (what every cell above already does via `.fetchdf()`) — good for "I want to keep working in this notebook."
- **`COPY ... TO` in SQL** — good for "I want a file on disk," and it's DuckDB writing the Parquet/CSV directly rather than round-tripping through pandas. (If you have `polars` installed, `.pl()` is the equivalent in-memory alternative to `.fetchdf()` — not used here to keep this notebook's dependencies to `duckdb` + `pandas`.)

In [7]:
import os

os.makedirs("../data", exist_ok=True)

conn.execute(f"""
    COPY (
        SELECT docket_id, agency_code, title, docket_type, modify_date
        FROM read_parquet('{R2_BASE_URL}/dockets.parquet')
        WHERE agency_code = 'EPA' AND title ILIKE '%PFAS%'
    ) TO '../data/epa_pfas_dockets.parquet' (FORMAT PARQUET)
""")
conn.execute(f"""
    COPY (
        SELECT docket_id, agency_code, title, docket_type, modify_date
        FROM read_parquet('{R2_BASE_URL}/dockets.parquet')
        WHERE agency_code = 'EPA' AND title ILIKE '%PFAS%'
    ) TO '../data/epa_pfas_dockets.csv' (FORMAT CSV, HEADER)
""")

# Read it back to confirm — both files are real, independent artifacts now.
conn.execute("SELECT COUNT(*) AS rows FROM read_parquet('../data/epa_pfas_dockets.parquet')").fetchdf()

,rows
0,19


**When to stop exporting one-off files and build a rollup instead:** if you find yourself re-running the same `COPY ... TO` on a schedule, or several people want the same derived table, that's the signal to add a proper rollup pipeline (`src/spicy_regs/pipelines/rollups/`) instead — decoupled, independently re-runnable, and published to R2 for everyone rather than living in someone's `../data/` folder. See the top-level project docs on adding an external-source or derived rollup.

## Where to go next

- **Search** (metadata + full-text) → [`search_capabilities.ipynb`](search_capabilities.ipynb)
- **Schema reference, agency explorer, export helpers** → [`data_explorer.ipynb`](data_explorer.ipynb)
- **The Iceberg catalog directly** (maintainer credentials) → [`iceberg_explorer.ipynb`](iceberg_explorer.ipynb)
- **Analysis tracks** (campaign detection, entity resolution, position/sentiment, influence mapping, docket/cross-docket analysis, document navigation) → see [`README.md`](README.md)